# Smart Batching Search - Testing Notebook

This notebook uses **[bigdata-smart-batching](https://pypi.org/project/bigdata-smart-batching/)** on PyPI: semantic search with intelligent company grouping, proportional sampling, and rate-limited parallel execution.

## Features
- **Planning**: Organize search using smart batching
- **Execution**: Execute search with proportional sampling
- **Rate Limiting**: Configurable requests per minute
- **Parallel Processing**: Efficient parallel execution

## Configuration

**Environment Variables:** This notebook loads configuration from a `.env` file in the `Smart_Batching` directory.

Create a `.env` file with:
```
BIGDATA_API_KEY=your_api_key_here
BIGDATA_API_BASE_URL=https://api.bigdata.com
```

**Options for API_BASE_URL:**
- `https://api.bigdata.com` (production - default)

**Note:** You must restart the kernel and run cells from the beginning if you change the API Base URL, as it's read at import time.

## 1. Load Environment Variables and Setup

**IMPORTANT:** Load `.env` file and set API_BASE_URL here before importing modules, as it's read at import time.

In [1]:
# Library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)

# Set API base URL BEFORE importing (smart_batching_config reads it at import time)
API_BASE_URL = os.getenv("BIGDATA_API_BASE_URL", "https://api.bigdata.com")
os.environ["BIGDATA_API_BASE_URL"] = API_BASE_URL

# Import utilities from bigdata-smart-batching package
from bigdata_smart_batching import (
    plan_search,
    execute_search,
    deduplicate_documents,
    save_plan,
    load_plan,
    load_universe_from_csv,
    convert_to_dataframe,
)

## 2. Configuration

In [2]:
# Configuration
# Note: API_BASE_URL and API_KEY are loaded from .env file in cell 2
# To change them, edit the .env file or set environment variables:
#   export BIGDATA_API_KEY='your_api_key_here'
#   export BIGDATA_API_BASE_URL='https://api.bigdata.com'

# API Key (loaded from .env in cell 2)
API_KEY = os.getenv("BIGDATA_API_KEY")

if not API_KEY:
    print("⚠️  BIGDATA_API_KEY not set. Please set it in your .env file:")
    print("   BIGDATA_API_KEY=your_api_key_here")
    print("   Or set environment variable: export BIGDATA_API_KEY='your_api_key_here'")
else:
    print(f"✅ API Key configured: {API_KEY[:8]}...{API_KEY[-4:]}")

# Test parameters
# SMOKE TEST SCOPE: right-sized to 10 companies / 30-day window for a fast,
# low-cost smoke pass. Swap TEST_UNIVERSE_CSV back to
# "id_name_mapping_us_top_3000.csv" (~3000 companies, id column) for a
# full-scale run.
TEST_TEXT = "The company has been impacted by strait of hormuz shipping disruption"
TEST_UNIVERSE_CSV = "../Thematic_Screener_CLI/40_companies.csv"
TEST_UNIVERSE_ID_COLUMN = "RP_ENTITY_ID"
TEST_UNIVERSE_SIZE = 10  # smoke test: first 10 companies only
TEST_START_DATE = "2026-07-01"
TEST_END_DATE = "2026-07-31"  # 30-day window
TEST_CHUNK_PERCENTAGE = 0.1  # 10% of total chunks

print(f"\n📝 Test Configuration:")
print(f"   API Base URL: {API_BASE_URL}")
print(f"   Text: '{TEST_TEXT}'")
print(f"   Universe: {TEST_UNIVERSE_CSV} (first {TEST_UNIVERSE_SIZE} companies)")
print(f"   Date Range: {TEST_START_DATE} to {TEST_END_DATE}")
print(f"   Chunk Percentage: {TEST_CHUNK_PERCENTAGE*100:.0f}%")

✅ API Key configured: bd_v2_Nq...nP78

📝 Test Configuration:
   API Base URL: https://api.bigdata.com
   Text: 'The company has been impacted by strait of hormuz shipping disruption'
   Universe: ../Thematic_Screener_CLI/40_companies.csv (first 10 companies)
   Date Range: 2026-07-01 to 2026-07-31
   Chunk Percentage: 10%


## 3. Test Universe Loading

In [3]:
# Test loading universe from CSV
# load_universe_from_csv() (and plan_search's internal universe resolver)
# default to an "id" column header. Our universe CSV uses RP_ENTITY_ID and we
# only want the first 10 companies for this smoke test, so we load it
# directly with pandas and pass a plain list[str] to plan_search below.
import pandas as pd

try:
    universe_df = pd.read_csv(TEST_UNIVERSE_CSV)
    companies = (
        universe_df[TEST_UNIVERSE_ID_COLUMN]
        .astype(str)
        .str.strip()
        .head(TEST_UNIVERSE_SIZE)
        .tolist()
    )
    print(f"✅ Loaded {len(companies)} companies from {TEST_UNIVERSE_CSV}")
    print(f"   First 5 companies: {companies[:5]}")
except Exception as e:
    print(f"❌ Error loading universe: {e}")

✅ Loaded 10 companies from ../Thematic_Screener_CLI/40_companies.csv
   First 5 companies: ['E09E2B', 'D8442A', '228D42', '0157B1', '4A6F00']


## 4. Step 1: Plan Search

In [4]:
# Plan the search
if API_KEY:
    print("📋 Planning search...")
    print("-" * 80)

    try:
        plan = plan_search(
            text=TEST_TEXT,
            universe=companies,  # smoke test: list of 10 RP_ENTITY_IDs (not the full CSV)
            start_date=TEST_START_DATE,
            end_date=TEST_END_DATE,
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
            volume_query_mode="iterative",
            max_iterations_per_batch=10
        )

        print(f"\n✅ Planning complete!")
        print(f"   Total expected chunks: {plan['chunk_upper_bound_estimate']:,}")
        print(f"   Number of baskets: {len(plan['baskets'])}")

        if plan.get('planning_metadata'):
            metadata = plan['planning_metadata']
            print(f"   Total companies: {metadata.get('total_companies', 'N/A')}")
            print(f"   Companies with chunks: {metadata.get('companies_with_chunks', 'N/A')}")
            print(f"   Uses smart batching: {metadata.get('uses_smart_batching', False)}")

        # Show example basket
        if plan['baskets']:
            example_basket = plan['baskets'][0]
            print(f"\n   Example Basket:")
            print(f"     Basket ID: {example_basket['basket_id']}")
            print(f"     Expected chunks: {example_basket['expected_chunks']}")
            print(f"     Companies: {len(example_basket['companies'])} companies")
            print(f"     Query text: '{example_basket['query']['text']}'")
            print(f"     Max chunks in query: {example_basket['query']['max_chunks']}")

        # Smart batching vs. naive one-query-per-company comparison.
        # Naive approach: search each company individually for TEST_TEXT
        # (1 query x N companies). Smart batching groups companies into
        # fewer baskets, each covered by one API call.
        naive_query_count = len(companies) * 1
        smart_basket_count = len(plan['baskets'])
        print(f"\n   📊 Smart batching vs. naive:")
        print(f"     Naive (company x query) queries: {naive_query_count}")
        print(f"     Smart batching baskets:          {smart_basket_count}")
        assert smart_basket_count < naive_query_count, (
            "Expected smart batching to produce fewer baskets than the naive "
            "per-company query count"
        )
        print(f"     ✅ Smart batching reduced queries by "
              f"{(1 - smart_basket_count / naive_query_count) * 100:.0f}%")

    except Exception as e:
        print(f"❌ Error during planning: {e}")
        import traceback
        traceback.print_exc()
        plan = None
else:
    print("⚠️  Skipping planning - API key not set")
    plan = None

📋 Planning search...
--------------------------------------------------------------------------------
2026-08-14 09:06:17,710 - INFO - Planning search for text: 'The company has been impacted by strait of hormuz shipping disruption'


2026-08-14 09:06:17,711 - INFO - Date range: 2026-07-01 to 2026-07-31


2026-08-14 09:06:17,712 - INFO - Using 10 entity IDs from inline list


2026-08-14 09:06:17,712 - INFO - Loaded 10 companies from universe


2026-08-14 09:06:17,714 - INFO - Using SmartBatchingPlanner with adaptive time-window splitting


2026-08-14 09:06:17,715 - INFO - Using 10 entity IDs from inline list


PHASE 1: Querying full period for all companies (2026-07-01 to 2026-07-31)
         Mode: iterative
    [ITERATIVE MODE] Querying 10 companies in 1 batches of 500 across 1 date sub-range(s) (1 work items, max_workers=1)
    Each batch iterates until no new companies are found (max 10 iterations)


      Batch 1/1, Iter 1: Found 10 new companies, 0 remaining
      Batch 1 complete: 10 found, 0 very_low, 1 iterations
    Completed 1 queries across 1 date sub-range(s). Found 10 companies with chunks > 0, 0 very_low

Phase 1 complete: 1 comention queries
Found 10 companies with chunks > 0

Created 3 entity groups (min_entities_per_group=1)
  Group 0: 1 companies, 790 total chunks, bucket=high
  Group 1: 5 companies, 1000 total chunks, bucket=medium
  Group 2: 4 companies, 216 total chunks, bucket=low
  Zero chunks: 0 companies

PHASE 2: Planning SMART configuration (per-group time splits)
  Group 0: 1 period (full_range), 1 basket
  Group 1: 1 period (full_range), 1 basket
  Group 2: 1 period (full_range), 1 basket
2026-08-14 09:06:19,179 - INFO - Planning complete: 2,006 expected chunks in 3 baskets



✅ Planning complete!
   Total expected chunks: 2,006
   Number of baskets: 3
   Total companies: 10
   Companies with chunks: 10
   Uses smart batching: True

   Example Basket:
     Basket ID: basket_0_high_20260701_20260731
     Expected chunks: 790
     Companies: 1 companies
     Query text: 'The company has been impacted by strait of hormuz shipping disruption'
     Max chunks in query: 790

   📊 Smart batching vs. naive:
     Naive (company x query) queries: 10
     Smart batching baskets:          3
     ✅ Smart batching reduced queries by 70%


## 5. Save Plan (Optional)

In [5]:
# Save plan for later use
if plan:
    plan_file = "test_search_plan.json"
    try:
        save_plan(plan, plan_file)
        print(f"✅ Plan saved to {plan_file}")
        print(f"   You can load it later with: plan = load_plan('{plan_file}')")
    except Exception as e:
        print(f"❌ Error saving plan: {e}")

2026-08-14 09:06:19,186 - INFO - Plan saved to test_search_plan.json


✅ Plan saved to test_search_plan.json
   You can load it later with: plan = load_plan('test_search_plan.json')


## 6. Step 2: Execute Search with Proportional Sampling

In [6]:
# Execute search with proportional sampling
if plan and API_KEY:
    print("🔍 Executing search...")
    print("-" * 80)
    
    try:
        results_raw = execute_search(
            search_plan=plan,
            chunk_percentage=TEST_CHUNK_PERCENTAGE,
            requests_per_minute=450,  # Rate limit
            basket_filtered_entities=True,
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
        )
        
        results = deduplicate_documents(results_raw)

        print(f"\n✅ Search complete!")
        print(f"   Retrieved {len(results):,} deduplicated chunks")
            
    except Exception as e:
        print(f"❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()
        results = []
else:
    print("⚠️  Skipping execution - plan or API key not available")
    results = []

🔍 Executing search...
--------------------------------------------------------------------------------
2026-08-14 09:06:19,193 - INFO - Executing search with 10.0% of chunks


2026-08-14 09:06:19,193 - INFO - Total maximum expected chunks: 200


2026-08-14 09:06:19,194 - INFO - Searching 3 baskets


2026-08-14 09:06:20,857 - INFO - Basket basket_2_low_20260701_20260731: Retrieved 21 documents with 21 chunks


2026-08-14 09:06:21,507 - INFO - Basket basket_1_medium_20260701_20260731: Retrieved 97 documents with 100 chunks


2026-08-14 09:06:21,900 - INFO - Basket basket_0_high_20260701_20260731: Retrieved 79 documents with 79 chunks


2026-08-14 09:06:21,908 - INFO - First pass complete: 197 documents with 200 chunks


2026-08-14 09:06:21,909 - INFO - Search complete: 197 documents with 200 chunks retrieved in 2.71s


2026-08-14 09:06:21,909 - INFO - Deduplicated: 166 unique documents from 197 total (chunks merged)



✅ Search complete!
   Retrieved 166 deduplicated chunks


## 7. Analyze Results

In [7]:
results[2]

{'id': '8FE14957FF8BCDB7BE2905653EAFC8E9',
 'headline': 'Stocks and bonds drop as mounting US-Iran tensions spook investors',
 'timestamp': '2026-07-13T05:13:46',
 'source': {'id': 'DA9FC6', 'name': 'Financial Times', 'rank': 'RANK_1'},
 'url': 'https://www.ft.com/content/54be872f-c0a2-46dd-9337-5cad9124e734',
 'chunks': [{'cnum': 3,
   'text': '"It\'s no longer clear that the US and Iran have the same ends in mind," said Brien.\nHostilities have escalated over control of the Strait of Hormuz in the past week, putting a ceasefire agreement between the two sides on the verge of collapse and risking a return to full-scale war.\n"The recent oil price rally, driven by tanker attacks and renewed US-Iran strikes, demonstrates how critical Hormuz flows remain for prices in the short term," Goldman Sachs analysts noted.\nOn Wall Street, semiconductor stocks, which have been volatile in recent weeks following a blistering rally at the start of the year, led the declines. Memory company Sandisk 

In [8]:
# Convert to DataFrame (exploded by chunk)
df = convert_to_dataframe(results)
df.head(20)

,date,doc_id,headline,source_id,source_name,source_rank,chunk_index,chunk_text,chunk_relevance,chunk_sentiment,entity_ids,detections,url,reporting_entities
0,2026-07-28,36D35F8249CEFD0042A8A9190CEC4992,Trump Downplays Russia-Iran Intelligence Claim...,5A5702,Benzinga,RANK_1,4,"Despite the pause in direct hostilities, shipp...",0.331142,-0.77,"[69345C, 49BBBC]","[{'id': 'AFCFF2', 'start': 524, 'end': 553, 't...",https://www.benzinga.com/node/60714419?utm_cam...,[]
1,2026-07-13,3BC10B2A4616F3BED81CA921844466DB,"Kospi falls 9%, SK Hynix stock dives as Iran c...",E5AA62,Yahoo! Finance,RANK_2,1,Global markets slid Monday after Iran declared...,0.287124,-0.65,"[69345C, 49BBBC]","[{'id': '69345C', 'start': 443, 'end': 465, 't...",https://finance.yahoo.com/markets/world-indice...,[]
2,2026-07-13,8FE14957FF8BCDB7BE2905653EAFC8E9,Stocks and bonds drop as mounting US-Iran tens...,DA9FC6,Financial Times,RANK_1,3,"""It's no longer clear that the US and Iran hav...",0.276955,-0.63,[49BBBC],"[{'id': 'C4F920', 'start': 503, 'end': 509, 't...",https://www.ft.com/content/54be872f-c0a2-46dd-...,[]
3,2026-07-07,3E73F042732DF793D4F291E2D1A86347,US futures edge lower as investors assess Sams...,E5AA62,Yahoo! Finance,RANK_2,5,Strait of Hormuz tensions return\nGeopolitical...,0.272757,-0.66,"[09DE1F, 69345C]","[{'id': '2DA120', 'start': 849, 'end': 857, 't...",https://finance.yahoo.com/markets/stocks/artic...,[]
4,2026-07-17,AD34A82890DF6F1557C8D0ECCAF6D1D9,KB Global Tracker+ - Oil Shock and HBM Concern...,2C2430,"KB Securities Co., Ltd.",RANK_1,2,- Amid an over 9% surge in oil prices followin...,0.230261,-0.42,"[69345C, 49BBBC]","[{'id': '3D4567', 'start': 268, 'end': 270, 't...",https://research.bluematrix.com/docs/pdf/09125...,[]
5,2026-07-09,A8E9CD524BBE311AFAEFE5AB620D670A,"US stocks collectively rose, the 'Magnificent ...",855C25,Ifeng,RANK_3,2,The Philadelphia Semiconductor Index rose by o...,0.225675,-0.65,[49BBBC],"[{'id': '439DC3', 'start': 223, 'end': 231, 't...",https://finance.ifeng.com/c/8uczKFAKVpJ,[]
6,2026-07-09,E9DFABA18A8CC5518103F45F7938FC12,Global Financial Market Focus - Geopolitical R...,D89336,"Yuanta Financial Holding Co., Limited",RANK_1,4,"US Stock Market Review\nAt the NATO summit, Tr...",0.212359,-0.15,[09DE1F],"[{'id': '7CC810', 'start': 35, 'end': 41, 'typ...",https://research.bluematrix.com/docs/pdf/97652...,[]
7,2026-07-02,12E49F6B1A8793ADB0D68032948EE6C7,The Daily Chase: U.S. decides against renewing...,7490C8,BNN Bloomberg,RANK_1,2,The price of oil is down for a third straight ...,0.191668,-0.35,[228D42],"[{'id': '885348', 'start': 480, 'end': 488, 't...",https://www.bnnbloomberg.ca/business/economics...,[]
8,2026-07-10,331B3F6674B54971A0CA19B4B434F035,"META surges 6%, Micron down over 2%, SK Hynix ...",4C2766,Eastmoney (东方财富网),RANK_2,3,"The Joint Maritime Information Center (JMIC), ...",0.186160,-0.11,[49BBBC],"[{'id': '0BB903', 'start': 952, 'end': 959, 't...",https://finance.eastmoney.com/a/20260710380216...,[]
9,2026-07-07,1D6D6EA18657BAA8D1F274B2793B894D,London stocks mixed amid weak tech but FTSE 10...,648085,AOL.com,RANK_2,2,"While Ipek Ozkardeskaya, senior analyst at Swi...",0.179773,-0.69,"[69345C, 49BBBC]","[{'id': '69345C', 'start': 348, 'end': 370, 't...",https://www.aol.co.uk/articles/london-stocks-m...,[]


In [9]:
top10 = df.sort_values(by="chunk_relevance", ascending=False).head(10)
print(top10)

           date                            doc_id  \
114  2026-07-30  2E5A7ED4857987D22610792A2858FD98   
21   2026-07-30  7C2205BA7EDF3FA3E5E482562C586EFA   
115  2026-07-09  FD5F1DFFF559EDBE727D23430AF94BD8   
116  2026-07-09  60466D6982892FD1D0B6AF081A9A8226   
22   2026-07-07  7E1165D611111F505A306C686F3AAB40   
24   2026-07-27  43B3BD52861B5FA0195C218ED8A2C96B   
0    2026-07-28  36D35F8249CEFD0042A8A9190CEC4992   
117  2026-07-07  5AC22561043DAAF36A1AB83057B30814   
25   2026-07-15  514BE58C8B60A5F1504CFC5306EB7AFA   
118  2026-07-21  7E958B204BF45C98975E6889A102521C   

                                              headline source_id source_name  \
114  The Strait of Hormuz is just one trade chokepo...    AA1167        CNBC   
21   US renews strikes on Iran as widening war pull...    2435A4         CNN   
115  Tanker traffic through Strait of Hormuz slows ...    AA1167        CNBC   
116  Ship traffic through Strait of Hormuz plunges ...    001588   Gulf News   
22   Hormuz Toll

In [10]:
top10

,date,doc_id,headline,source_id,source_name,source_rank,chunk_index,chunk_text,chunk_relevance,chunk_sentiment,entity_ids,detections,url,reporting_entities
114,2026-07-30,2E5A7ED4857987D22610792A2858FD98,The Strait of Hormuz is just one trade chokepo...,AA1167,CNBC,RANK_1,9,"""Given the challenging maritime security envir...",0.479871,-0.51,[4A6F00],"[{'id': '555EB5', 'start': 180, 'end': 185, 't...",https://www.cnbc.com/2026/07/30/strait-hormuz-...,[]
21,2026-07-30,7C2205BA7EDF3FA3E5E482562C586EFA,US renews strikes on Iran as widening war pull...,2435A4,CNN,RANK_1,5,MarineTraffic data on Wednesday showed at leas...,0.443302,-0.38,[D8442A],"[{'id': '912D5E', 'start': 665, 'end': 673, 't...",https://www.cnn.com/2026/07/30/world/live-news...,[]
115,2026-07-09,FD5F1DFFF559EDBE727D23430AF94BD8,Tanker traffic through Strait of Hormuz slows ...,AA1167,CNBC,RANK_1,3,But the oil market is not pricing in a complet...,0.383941,-0.61,[4A6F00],"[{'id': '364C20', 'start': 1220, 'end': 1230, ...",https://www.cnbc.com/2026/07/09/iran-strait-ho...,[]
116,2026-07-09,60466D6982892FD1D0B6AF081A9A8226,Ship traffic through Strait of Hormuz plunges ...,001588,Gulf News,RANK_2,1,Ship traffic through Strait of Hormuz plunges ...,0.361706,-0.63,[4A6F00],"[{'id': '912D5E', 'start': 87, 'end': 95, 'typ...",https://gulfnews.com/business/energy/ship-traf...,[]
22,2026-07-07,7E1165D611111F505A306C686F3AAB40,Hormuz Tolls Are Just the Beginning: The World...,648085,AOL.com,RANK_2,2,How Hormuz Rewrote the Playbook\nThe trigger w...,0.336456,-0.74,[E09E2B],"[{'id': '9DBA1F', 'start': 573, 'end': 579, 't...",https://www.aol.com/articles/hormuz-tolls-just...,[]
24,2026-07-27,43B3BD52861B5FA0195C218ED8A2C96B,US-Iran Tensions Lead to Sluggish Shipping in ...,A15E96,Chinatimes,RANK_1,1,Shipping data tracking firm Kpler released dat...,0.335682,-0.56,[D8442A],"[{'id': '387806', 'start': 1355, 'end': 1369, ...",https://www.chinatimes.com/realtimenews/202607...,[]
0,2026-07-28,36D35F8249CEFD0042A8A9190CEC4992,Trump Downplays Russia-Iran Intelligence Claim...,5A5702,Benzinga,RANK_1,4,"Despite the pause in direct hostilities, shipp...",0.331142,-0.77,"[69345C, 49BBBC]","[{'id': 'AFCFF2', 'start': 524, 'end': 553, 't...",https://www.benzinga.com/node/60714419?utm_cam...,[]
117,2026-07-07,5AC22561043DAAF36A1AB83057B30814,Strait of Hormuz threat level raised to 'sever...,AA1167,CNBC,RANK_1,3,Hormuz has fractured into separate corridors c...,0.320764,-0.68,[4A6F00],"[{'id': '317F12', 'start': 168, 'end': 177, 't...",https://www.cnbc.com/2026/07/07/iran-strait-ho...,[]
25,2026-07-15,514BE58C8B60A5F1504CFC5306EB7AFA,Trump's protection fails? Hormuz increasingly ...,A15E96,Chinatimes,RANK_1,5,After the US Navy launched a Washington-coordi...,0.320678,-0.65,[D8442A],"[{'id': '912D5E', 'start': 81, 'end': 89, 'typ...",https://www.chinatimes.com/realtimenews/202607...,[]
118,2026-07-21,7E958B204BF45C98975E6889A102521C,Ships shun Strait of Hormuz as renewed fightin...,AA1167,CNBC,RANK_1,4,S&P Global data painted a similar picture. Jus...,0.304523,-0.35,[4A6F00],"[{'id': '940A72', 'start': 1030, 'end': 1034, ...",https://www.cnbc.com/2026/07/21/strait-of-horm...,[]


In [11]:
for i, headline in enumerate(top10["chunk_text"], start=1):
    print(f"{i}. {headline}")

1. "Given the challenging maritime security environment, rates have increased from levels that owners and charterers will be used to. The cost will vary depending on the vessel type, cargo and routing, however marine insurers are continuing to provide cover and helping to ensure marine commerce can continue with adequate coverage in place," they added.
How companies are responding to shipping risks
Bejjani told CNBC that the structural consequence of maritime warfare "is bigger than most people realize."
"The Gulf is bracketed by two straits, not one, and the region is now designing around both Hormuz and Bab el-Mandeb to the maximum extent possible," he said.
"That is new. Past crises produced hedges. This one is producing an architecture: overland corridors, bypass pipelines, forward storage near the markets that matter most. It will cost heavily, take a decade, and ripple for decades more. I expect other strait-dependent regions to follow, though few with the same urgency or resourc

In [12]:
# entity_ids holds a list per row, which pandas can't hash directly for
# groupby, so we group on the tuple form of each row's entity ID combination.
source_counts = df.groupby(df['entity_ids'].apply(tuple)).size().sort_values(ascending=False)

In [13]:
#Analyze results
if results:

    print("📈 Results Analysis")
    print("-" * 80)
    
    # Summary stats
    n_docs = df['doc_id'].nunique()
    n_chunks = len(df)
    print(f"\n   Total: {n_docs:,} documents, {n_chunks:,} chunks")
    
    # Relevance distribution
    if 'chunk_relevance' in df.columns and df['chunk_relevance'].notna().any():
        print(f"\n   Relevance Scores:")
        print(f"     Min: {df['chunk_relevance'].min():.3f}")
        print(f"     Max: {df['chunk_relevance'].max():.3f}")
        print(f"     Avg: {df['chunk_relevance'].mean():.3f}")
    
    # Sentiment distribution
    if 'chunk_sentiment' in df.columns and df['chunk_sentiment'].notna().any():
        sentiments = df['chunk_sentiment'].dropna()
        positive = (sentiments > 0).sum()
        negative = (sentiments < 0).sum()
        neutral = len(sentiments) - positive - negative
        print(f"\n   Sentiment Distribution:")
        print(f"     Positive: {positive} ({positive/len(sentiments)*100:.1f}%)")
        print(f"     Negative: {negative} ({negative/len(sentiments)*100:.1f}%)")
        print(f"     Neutral: {neutral} ({neutral/len(sentiments)*100:.1f}%)")
    
    # Source distribution
    if 'source_name' in df.columns:
        source_counts = df.groupby('source_name').size().sort_values(ascending=False)
        print(f"\n   Top Sources:")
        for source, count in source_counts.head(5).items():
            print(f"     {source}: {count} chunks")
    
    # Show DataFrame info
    print(f"\n   DataFrame shape: {df.shape}")
    
    # Save results
    from datetime import datetime
    results_file = f"search_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    df.to_json(results_file, orient='records', indent=2)
    print(f"\n💾 Saved to {results_file}")

else:
    df = None
    print("⚠️  No results to analyze")

📈 Results Analysis
--------------------------------------------------------------------------------

   Total: 166 documents, 169 chunks

   Relevance Scores:
     Min: 0.039
     Max: 0.480
     Avg: 0.144

   Sentiment Distribution:
     Positive: 12 (7.1%)
     Negative: 157 (92.9%)
     Neutral: 0 (0.0%)

   Top Sources:
     Chinatimes: 29 chunks
     CNBC: 26 chunks
     Yahoo! Finance: 11 chunks
     MT Newswires: 8 chunks
     AOL.com: 6 chunks

   DataFrame shape: (169, 14)

💾 Saved to search_results_20260814_090622.json


## 8. Summary

In [14]:
print("=" * 80)
print("Smart Batching Search - Test Summary")
print("=" * 80)

if plan:
    print(f"✅ Planning: SUCCESS")
    print(f"   Expected chunks: {plan['chunk_upper_bound_estimate']:,}")
    print(f"   Baskets created: {len(plan['baskets'])}")
else:
    print("⚠️  Planning: Not completed")

if results:
    print(f"✅ Execution: SUCCESS")
    print(f"   Chunks retrieved: {len(results):,}")
    print(f"   Percentage used: {TEST_CHUNK_PERCENTAGE*100:.0f}%")
    if plan:
        expected = plan['chunk_upper_bound_estimate']
        actual = len(results)
        if expected > 0:
            print(f"   Actual vs Expected: {actual/expected*100:.1f}%")
else:
    print("⚠️  Execution: Not completed")

print("\n" + "=" * 80)
print("Test complete!")
print("=" * 80)

Smart Batching Search - Test Summary
✅ Planning: SUCCESS
   Expected chunks: 2,006
   Baskets created: 3
✅ Execution: SUCCESS
   Chunks retrieved: 166
   Percentage used: 10%
   Actual vs Expected: 8.3%

Test complete!


## 9. Load Saved Plan (Optional)

In [15]:
# Load a previously saved plan
plan_file = "test_search_plan.json"

if os.path.exists(plan_file):
    try:
        loaded_plan = load_plan(plan_file)
        print(f"✅ Plan loaded from {plan_file}")
        print(f"   Total expected chunks: {loaded_plan.get('chunk_upper_bound_estimate', 0):,}")
        print(f"   Number of baskets: {len(loaded_plan.get('baskets', []))}")
        print(f"\n   You can now execute with different percentages:")
        print(f"   results = execute_search(loaded_plan, chunk_percentage=0.2)")
    except Exception as e:
        print(f"❌ Error loading plan: {e}")
else:
    print(f"ℹ️  Plan file '{plan_file}' not found. Save a plan first.")

2026-08-14 09:06:22,165 - INFO - Plan loaded from test_search_plan.json


✅ Plan loaded from test_search_plan.json
   Total expected chunks: 2,006
   Number of baskets: 3

   You can now execute with different percentages:
   results = execute_search(loaded_plan, chunk_percentage=0.2)
